# JOIN — propojování tabulek

V předchozích noteboocích jsme pracovali vždy s **jednou tabulkou**. V praxi jsou ale data rozdělena do **více tabulek**, které spolu souvisejí (např. hráči a týmy). Abychom mohli zobrazit data z více tabulek najednou, použijeme klíčové slovo `JOIN`.

---

## Přehled typů JOIN

| Typ | Popis | Výsledek |
|---|---|---|
| `INNER JOIN` | Pouze řádky, které mají **shodu v obou** tabulkách | Žádné NULL z JOINu |
| `LEFT JOIN` | Všechny řádky z **levé** tabulky + shody z pravé | NULL z pravé tabulky, pokud shoda chybí |
| `RIGHT JOIN` | Všechny řádky z **pravé** tabulky + shody z levé | NULL z levé tabulky, pokud shoda chybí |
| `FULL OUTER JOIN` | Všechny řádky z **obou** tabulek | NULL na obou stranách, kde shoda chybí |

### Syntaxe

```sql
SELECT sloupce
FROM tabulka_a
JOIN tabulka_b ON tabulka_a.sloupec = tabulka_b.sloupec;
```

- **Levá tabulka** = ta před slovem `JOIN` (v klauzuli `FROM`)
- **Pravá tabulka** = ta za slovem `JOIN`
- **ON** = podmínka, podle které se řádky párují (typicky FK = PK)

> **Tip:** Pokud mají sloupce v různých tabulkách stejný název, je nutné je rozlišit pomocí **tečkové notace**: `tabulka.sloupec`

## Instalace balíčku

In [ ]:
# Instalace knihovny (stačí spustit jednou)
! pip install mysql-connector-python

## Připojení k databázi

In [ ]:
import mysql.connector
from pripojeni import *  # importuje konstanty HOST, USER, PASSWORD, DATABASE

mydb = mysql.connector.connect(
    host=HOST,
    user=USER,
    password=PASSWORD,
    database=DATABASE
)

mycursor = mydb.cursor()

## Příprava ukázkových dat

Vytvoříme si 3 propojené tabulky — `tymy`, `pozice` a `hraci`:

```
tymy (1) ◄──── (N) hraci (N) ────► (1) pozice
```

Každý hráč patří do **jednoho týmu** (`tym_id`) a hraje na **jedné pozici** (`pozice_id`).

Záměrně vložíme i hráče s **neexistujícím** `tym_id` a `pozice_id`, abychom viděli rozdíly mezi typy JOIN.

In [ ]:
# Úklid starých tabulek (pořadí kvůli závislostem)
mycursor.execute("DROP TABLE IF EXISTS hraci")
mycursor.execute("DROP TABLE IF EXISTS pozice")
mycursor.execute("DROP TABLE IF EXISTS tymy")

# Tabulka týmů
mycursor.execute("""
    CREATE TABLE tymy (
        id INT PRIMARY KEY AUTO_INCREMENT,
        nazev VARCHAR(30) NOT NULL,
        mesto VARCHAR(30) NOT NULL,
        zustatek INT DEFAULT 0,
        CONSTRAINT unikatni_tym UNIQUE (nazev, mesto)
    )
""")

# Tabulka pozic
mycursor.execute("""
    CREATE TABLE pozice (
        id INT PRIMARY KEY AUTO_INCREMENT,
        pojmenovani VARCHAR(20) NOT NULL
    )
""")

# Tabulka hráčů (bez FOREIGN KEY — záměrně, abychom mohli vložit neplatné hodnoty)
mycursor.execute("""
    CREATE TABLE hraci (
        id INT PRIMARY KEY AUTO_INCREMENT,
        jmeno VARCHAR(30) NOT NULL,
        prijmeni VARCHAR(30) NOT NULL,
        datum_narozeni DATE NOT NULL,
        tym_id INT NOT NULL,
        pozice_id INT NOT NULL
    )
""")

# Naplnění dat
mycursor.execute("""
    INSERT INTO tymy (nazev, mesto, zustatek) VALUES
        ('HC Rytiri', 'Kladno', 1000),
        ('HC Slavie', 'Praha',  5000),
        ('HC Sparta', 'Praha',  6000),
        ('HC Sparta', 'Brno',   5600)
""")

mycursor.execute("""
    INSERT INTO pozice (pojmenovani) VALUES
        ('Utocnik'),
        ('Obrana'),
        ('Brankar'),
        ('Zaloha')
""")

mycursor.execute("""
    INSERT INTO hraci (jmeno, prijmeni, datum_narozeni, tym_id, pozice_id) VALUES
        ('Adam',  'Prvni',   '1996-06-02', 2, 2),
        ('Pavel', 'Druhy',   '2000-09-09', 1, 3),
        ('Kuba',  'Treti',   '1998-01-05', 3, 1),
        ('Josef', 'Ctvrty',  '1997-01-06', 6, 2),
        ('Jirka', 'Paty',    '1999-03-08', 2, 6)
""")

mydb.commit()
print("✅ Tabulky 'tymy', 'pozice' a 'hraci' vytvořeny a naplněny.")
print("   Hráč 'Ctvrty' má tym_id=6 (neexistuje) — uvidíme ho jen v RIGHT/FULL JOIN.")
print("   Hráč 'Paty' má pozice_id=6 (neexistuje) — uvidíme ho jen v LEFT/FULL JOIN.")

---

# INNER JOIN

`INNER JOIN` (nebo zkráceně jen `JOIN`) vrátí pouze řádky, které mají **shodu v obou tabulkách**. Řádky bez shody se **nezobrazí**.

```sql
SELECT sloupce
FROM tabulka_a
INNER JOIN tabulka_b ON tabulka_a.fk = tabulka_b.pk;
```

> **Pozn.:** `INNER JOIN` je v MySQL výchozí — klíčové slovo `INNER` lze vynechat.

### Příklad — JOIN bez podmínky (kartézský součin ❌)

In [ ]:
# Bez ON — vznikne kartézský součin (každý hráč × každý tým)
mycursor.execute("""
    SELECT prijmeni, tymy.nazev, tymy.mesto
    FROM hraci
    INNER JOIN tymy
""")

print(f"Kartézský součin: {len(mycursor.fetchall())} řádků (5 hráčů × 4 týmy = 20)")
print("⛔ Toto je téměř vždy CHYBA — chybí klauzule ON!")

### Příklad — INNER JOIN s podmínkou (✅ správně)

In [ ]:
# Správný JOIN — propojíme hráče s týmy přes tym_id = id
mycursor.execute("""
    SELECT hraci.prijmeni, hraci.datum_narozeni, tymy.nazev, tymy.mesto
    FROM hraci
    JOIN tymy ON hraci.tym_id = tymy.id
""")

print("INNER JOIN (hraci + tymy):")
for prijmeni, narozen, nazev, mesto in mycursor.fetchall():
    print(f"  {prijmeni} (nar. {narozen}) → {nazev}, {mesto}")

# ☝️ Hráč 'Ctvrty' (tym_id=6) a 'Paty' se zobrazí/nezobrazí podle toho,
# zda má odpovídající záznam v tabulce tymy
print("\n→ 'Ctvrty' chybí — jeho tym_id=6 nemá odpovídající tým.")

---

# LEFT JOIN

`LEFT JOIN` vrátí **všechny řádky z levé tabulky** (před JOIN). Pokud k řádku neexistuje shoda v pravé tabulce, doplní se `NULL`.

```sql
SELECT sloupce
FROM leva_tabulka
LEFT JOIN prava_tabulka ON leva.fk = prava.pk;
```

| Co je vlevo | Co je vpravo | Výsledek |
|---|---|---|
| `hraci` | `tymy` | Všichni hráči, i ti bez platného týmu |
| `tymy` | `hraci` | Všechny týmy, i ty bez hráčů |

### Příklad — všechny týmy + jejich hráči

In [ ]:
# Levá tabulka = tymy → zobrazí se VŠECHNY týmy
# Pokud tým nemá žádného hráče, sloupce z tabulky hraci budou NULL
mycursor.execute("""
    SELECT tymy.nazev, tymy.mesto, hraci.prijmeni, hraci.datum_narozeni
    FROM tymy
    LEFT JOIN hraci ON tymy.id = hraci.tym_id
""")

print("LEFT JOIN (tymy + hraci):")
for nazev, mesto, prijmeni, narozen in mycursor.fetchall():
    hrac = f"{prijmeni} (nar. {narozen})" if prijmeni else "— žádný hráč —"
    print(f"  {nazev}, {mesto}: {hrac}")

print("\n→ 'HC Sparta Brno' nemá žádného hráče — přesto se zobrazí (s NULL).")

---

# RIGHT JOIN

`RIGHT JOIN` vrátí **všechny řádky z pravé tabulky** (za JOIN). Pokud k řádku neexistuje shoda v levé tabulce, doplní se `NULL`.

```sql
SELECT sloupce
FROM leva_tabulka
RIGHT JOIN prava_tabulka ON leva.fk = prava.pk;
```

> **Tip:** `RIGHT JOIN` lze vždy přepsat jako `LEFT JOIN` s prohozenými tabulkami. V praxi se používá méně často.

### Příklad — všichni hráči + jejich tým

In [ ]:
# Pravá tabulka = hraci → zobrazí se VŠICHNI hráči
# Pokud hráč nemá platný tým, sloupce z tabulky tymy budou NULL
mycursor.execute("""
    SELECT tymy.nazev, tymy.mesto, hraci.prijmeni, hraci.datum_narozeni
    FROM tymy
    RIGHT JOIN hraci ON tymy.id = hraci.tym_id
""")

print("RIGHT JOIN (tymy + hraci):")
for nazev, mesto, prijmeni, narozen in mycursor.fetchall():
    tym = f"{nazev}, {mesto}" if nazev else "— bez týmu —"
    print(f"  {prijmeni} (nar. {narozen}) → {tym}")

print("\n→ 'Ctvrty' (tym_id=6) se zobrazí — jeho tým je NULL.")

---

# FULL OUTER JOIN

`FULL OUTER JOIN` vrátí **všechny řádky z obou tabulek**. Jedná se o kombinaci `LEFT JOIN` a `RIGHT JOIN`.

- Řádky se shodou → spojí se.
- Řádky bez shody vlevo → NULL v pravých sloupcích.
- Řádky bez shody vpravo → NULL v levých sloupcích.

> **⚠️ Pozor:** MySQL **nepodporuje** syntaxi `FULL OUTER JOIN`. Musíme ho obejít pomocí `UNION` — spojíme výsledky `LEFT JOIN` a `RIGHT JOIN`.

```sql
SELECT sloupce FROM tabulka_a LEFT JOIN tabulka_b ON ...
UNION
SELECT sloupce FROM tabulka_a RIGHT JOIN tabulka_b ON ...;
```

> **Pozn.:** `UNION` automaticky odstraní duplicitní řádky. Pokud chcete zachovat i duplicity, použijte `UNION ALL`.

### Příklad

In [ ]:
# FULL OUTER JOIN emulovaný přes UNION
mycursor.execute("""
    SELECT tymy.nazev, tymy.mesto, hraci.prijmeni, hraci.datum_narozeni
    FROM tymy
    LEFT JOIN hraci ON tymy.id = hraci.tym_id
    UNION
    SELECT tymy.nazev, tymy.mesto, hraci.prijmeni, hraci.datum_narozeni
    FROM tymy
    RIGHT JOIN hraci ON tymy.id = hraci.tym_id
""")

print("FULL OUTER JOIN (tymy + hraci):")
for nazev, mesto, prijmeni, narozen in mycursor.fetchall():
    tym = f"{nazev}, {mesto}" if nazev else "— bez týmu —"
    hrac = f"{prijmeni} (nar. {narozen})" if prijmeni else "— žádný hráč —"
    print(f"  {tym}: {hrac}")

print("\n→ Vidíme i tým bez hráče (Sparta Brno) i hráče bez týmu (Ctvrty).")

---

# Srovnání typů JOIN

Na našich datech — hráč `Ctvrty` má `tym_id=6` (neexistuje), tým `HC Sparta Brno` nemá žádného hráče:

| Typ JOIN | Ctvrty (bez týmu) | Sparta Brno (bez hráče) |
|---|---|---|
| `INNER JOIN` | ❌ Nezobrazí se | ❌ Nezobrazí se |
| `LEFT JOIN` (tymy vlevo) | ❌ Nezobrazí se | ✅ Zobrazí se (hráč=NULL) |
| `RIGHT JOIN` (hraci vpravo) | ✅ Zobrazí se (tým=NULL) | ❌ Nezobrazí se |
| `FULL OUTER JOIN` | ✅ Zobrazí se | ✅ Zobrazí se |

---

# JOIN tří tabulek

Pokud potřebujeme data z **více než dvou tabulek**, jednoduše zřetězíme další `JOIN`:

```sql
SELECT sloupce
FROM tabulka_a
JOIN tabulka_b ON tabulka_a.fk1 = tabulka_b.pk
JOIN tabulka_c ON tabulka_a.fk2 = tabulka_c.pk;
```

V našem případě chceme k hráčům připojit **název týmu** i **název pozice**:

```
tymy ◄── hraci ──► pozice
```

### Příklad

In [ ]:
# JOIN tří tabulek — hráči + týmy + pozice
mycursor.execute("""
    SELECT tymy.nazev, tymy.mesto, hraci.prijmeni, pozice.pojmenovani
    FROM tymy
    JOIN hraci ON tymy.id = hraci.tym_id
    JOIN pozice ON hraci.pozice_id = pozice.id
""")

print("JOIN tří tabulek (tymy + hraci + pozice):")
for nazev, mesto, prijmeni, poz in mycursor.fetchall():
    print(f"  {prijmeni} → {nazev} ({mesto}), pozice: {poz}")

print("\n→ Zobrazí se pouze hráči, kteří mají platný tým I platnou pozici.")

## Odpojení a úklid

In [ ]:
mycursor.execute("DROP TABLE IF EXISTS hraci")
mycursor.execute("DROP TABLE IF EXISTS pozice")
mycursor.execute("DROP TABLE IF EXISTS tymy")
mydb.commit()

mycursor.close()
mydb.close()

print("✅ Odpojení proběhlo úspěšně.")

---

# Cvičení

V každém cvičení jsou tabulky `tymy`, `pozice` a `hraci` předvytvořeny se stejnými daty jako v ukázkové části.

> Neupravujte přípravný kód. Na konci se vždy odpojte.

## Cvičení 1 — INNER JOIN

Vypište `jmeno`, `prijmeni` a `pojmenovani` (název pozice) hráčů.

Použijte `INNER JOIN` mezi tabulkami `hraci` a `pozice` (propojení přes `pozice_id = id`).

Zobrazí se pouze hráči, kteří mají **platnou pozici**.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Adam Prvni — Obrana
Pavel Druhy — Brankar
Kuba Treti — Utocnik
Josef Ctvrty — Obrana
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS hraci")
mycursor.execute("DROP TABLE IF EXISTS pozice")
mycursor.execute("DROP TABLE IF EXISTS tymy")
mycursor.execute("""
    CREATE TABLE tymy (
        id INT PRIMARY KEY AUTO_INCREMENT,
        nazev VARCHAR(30) NOT NULL,
        mesto VARCHAR(30) NOT NULL,
        zustatek INT DEFAULT 0
    )
""")
mycursor.execute("""
    CREATE TABLE pozice (
        id INT PRIMARY KEY AUTO_INCREMENT,
        pojmenovani VARCHAR(20) NOT NULL
    )
""")
mycursor.execute("""
    CREATE TABLE hraci (
        id INT PRIMARY KEY AUTO_INCREMENT,
        jmeno VARCHAR(30) NOT NULL,
        prijmeni VARCHAR(30) NOT NULL,
        datum_narozeni DATE NOT NULL,
        tym_id INT NOT NULL,
        pozice_id INT NOT NULL
    )
""")
mycursor.execute("""
    INSERT INTO tymy (nazev, mesto, zustatek) VALUES
        ('HC Rytiri', 'Kladno', 1000), ('HC Slavie', 'Praha', 5000),
        ('HC Sparta', 'Praha', 6000), ('HC Sparta', 'Brno', 5600)
""")
mycursor.execute("""
    INSERT INTO pozice (pojmenovani) VALUES
        ('Utocnik'), ('Obrana'), ('Brankar'), ('Zaloha')
""")
mycursor.execute("""
    INSERT INTO hraci (jmeno, prijmeni, datum_narozeni, tym_id, pozice_id) VALUES
        ('Adam','Prvni','1996-06-02',2,2), ('Pavel','Druhy','2000-09-09',1,3),
        ('Kuba','Treti','1998-01-05',3,1), ('Josef','Ctvrty','1997-01-06',6,2),
        ('Jirka','Paty','1999-03-08',2,6)
""")
mydb.commit()

# TODO: INNER JOIN hraci + pozice → vypište jmeno, prijmeni, pojmenovani →


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE hraci")
mycursor.execute("DROP TABLE pozice")
mycursor.execute("DROP TABLE tymy")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 2 — LEFT JOIN

Vypište **všechny hráče** a u těch, kteří mají platnou pozici, vypište i název pozice.

Vypište `jmeno`, `prijmeni`, `pojmenovani`.

> **Nápověda:** Levá tabulka musí být `hraci`, aby se zobrazili všichni.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Adam Prvni — Obrana
Pavel Druhy — Brankar
Kuba Treti — Utocnik
Josef Ctvrty — Obrana
Jirka Paty — None
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS hraci")
mycursor.execute("DROP TABLE IF EXISTS pozice")
mycursor.execute("DROP TABLE IF EXISTS tymy")
mycursor.execute("""
    CREATE TABLE tymy (
        id INT PRIMARY KEY AUTO_INCREMENT,
        nazev VARCHAR(30) NOT NULL,
        mesto VARCHAR(30) NOT NULL,
        zustatek INT DEFAULT 0
    )
""")
mycursor.execute("""
    CREATE TABLE pozice (
        id INT PRIMARY KEY AUTO_INCREMENT,
        pojmenovani VARCHAR(20) NOT NULL
    )
""")
mycursor.execute("""
    CREATE TABLE hraci (
        id INT PRIMARY KEY AUTO_INCREMENT,
        jmeno VARCHAR(30) NOT NULL,
        prijmeni VARCHAR(30) NOT NULL,
        datum_narozeni DATE NOT NULL,
        tym_id INT NOT NULL,
        pozice_id INT NOT NULL
    )
""")
mycursor.execute("""
    INSERT INTO tymy (nazev, mesto, zustatek) VALUES
        ('HC Rytiri', 'Kladno', 1000), ('HC Slavie', 'Praha', 5000),
        ('HC Sparta', 'Praha', 6000), ('HC Sparta', 'Brno', 5600)
""")
mycursor.execute("""
    INSERT INTO pozice (pojmenovani) VALUES
        ('Utocnik'), ('Obrana'), ('Brankar'), ('Zaloha')
""")
mycursor.execute("""
    INSERT INTO hraci (jmeno, prijmeni, datum_narozeni, tym_id, pozice_id) VALUES
        ('Adam','Prvni','1996-06-02',2,2), ('Pavel','Druhy','2000-09-09',1,3),
        ('Kuba','Treti','1998-01-05',3,1), ('Josef','Ctvrty','1997-01-06',6,2),
        ('Jirka','Paty','1999-03-08',2,6)
""")
mydb.commit()

# TODO: LEFT JOIN → všichni hráči + jejich pozice (i ti bez platné pozice) →


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE hraci")
mycursor.execute("DROP TABLE pozice")
mycursor.execute("DROP TABLE tymy")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 3 — RIGHT JOIN

Vypište **všechny pozice** a u obsazených pozic vypište jméno a příjmení hráče.

Vypište `jmeno`, `prijmeni`, `pojmenovani`.

> **Nápověda:** Pravá tabulka musí být `pozice`, aby se zobrazily všechny pozice.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Kuba Treti — Utocnik
Adam Prvni — Obrana
Josef Ctvrty — Obrana
Pavel Druhy — Brankar
None None — Zaloha
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS hraci")
mycursor.execute("DROP TABLE IF EXISTS pozice")
mycursor.execute("DROP TABLE IF EXISTS tymy")
mycursor.execute("""
    CREATE TABLE tymy (
        id INT PRIMARY KEY AUTO_INCREMENT,
        nazev VARCHAR(30) NOT NULL,
        mesto VARCHAR(30) NOT NULL,
        zustatek INT DEFAULT 0
    )
""")
mycursor.execute("""
    CREATE TABLE pozice (
        id INT PRIMARY KEY AUTO_INCREMENT,
        pojmenovani VARCHAR(20) NOT NULL
    )
""")
mycursor.execute("""
    CREATE TABLE hraci (
        id INT PRIMARY KEY AUTO_INCREMENT,
        jmeno VARCHAR(30) NOT NULL,
        prijmeni VARCHAR(30) NOT NULL,
        datum_narozeni DATE NOT NULL,
        tym_id INT NOT NULL,
        pozice_id INT NOT NULL
    )
""")
mycursor.execute("""
    INSERT INTO tymy (nazev, mesto, zustatek) VALUES
        ('HC Rytiri', 'Kladno', 1000), ('HC Slavie', 'Praha', 5000),
        ('HC Sparta', 'Praha', 6000), ('HC Sparta', 'Brno', 5600)
""")
mycursor.execute("""
    INSERT INTO pozice (pojmenovani) VALUES
        ('Utocnik'), ('Obrana'), ('Brankar'), ('Zaloha')
""")
mycursor.execute("""
    INSERT INTO hraci (jmeno, prijmeni, datum_narozeni, tym_id, pozice_id) VALUES
        ('Adam','Prvni','1996-06-02',2,2), ('Pavel','Druhy','2000-09-09',1,3),
        ('Kuba','Treti','1998-01-05',3,1), ('Josef','Ctvrty','1997-01-06',6,2),
        ('Jirka','Paty','1999-03-08',2,6)
""")
mydb.commit()

# TODO: RIGHT JOIN → všechny pozice + hráči na nich →


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE hraci")
mycursor.execute("DROP TABLE pozice")
mycursor.execute("DROP TABLE tymy")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 4 — FULL OUTER JOIN

Vypište **všechny hráče** a **všechny pozice** pomocí emulovaného `FULL OUTER JOIN` (LEFT + RIGHT + UNION).

Vypište **všechny sloupce** z obou tabulek.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Hráč #1 Adam Prvni (nar. 1996-06-02, tým=2, poz=2) — Pozice #2 Obrana
Hráč #2 Pavel Druhy (nar. 2000-09-09, tým=1, poz=3) — Pozice #3 Brankar
Hráč #3 Kuba Treti (nar. 1998-01-05, tým=3, poz=1) — Pozice #1 Utocnik
Hráč #4 Josef Ctvrty (nar. 1997-01-06, tým=6, poz=2) — Pozice #2 Obrana
Hráč #5 Jirka Paty (nar. 1999-03-08, tým=2, poz=6) — Pozice #None None
Hráč #None None None (nar. None, tým=None, poz=None) — Pozice #4 Zaloha
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS hraci")
mycursor.execute("DROP TABLE IF EXISTS pozice")
mycursor.execute("DROP TABLE IF EXISTS tymy")
mycursor.execute("""
    CREATE TABLE tymy (
        id INT PRIMARY KEY AUTO_INCREMENT,
        nazev VARCHAR(30) NOT NULL,
        mesto VARCHAR(30) NOT NULL,
        zustatek INT DEFAULT 0
    )
""")
mycursor.execute("""
    CREATE TABLE pozice (
        id INT PRIMARY KEY AUTO_INCREMENT,
        pojmenovani VARCHAR(20) NOT NULL
    )
""")
mycursor.execute("""
    CREATE TABLE hraci (
        id INT PRIMARY KEY AUTO_INCREMENT,
        jmeno VARCHAR(30) NOT NULL,
        prijmeni VARCHAR(30) NOT NULL,
        datum_narozeni DATE NOT NULL,
        tym_id INT NOT NULL,
        pozice_id INT NOT NULL
    )
""")
mycursor.execute("""
    INSERT INTO tymy (nazev, mesto, zustatek) VALUES
        ('HC Rytiri', 'Kladno', 1000), ('HC Slavie', 'Praha', 5000),
        ('HC Sparta', 'Praha', 6000), ('HC Sparta', 'Brno', 5600)
""")
mycursor.execute("""
    INSERT INTO pozice (pojmenovani) VALUES
        ('Utocnik'), ('Obrana'), ('Brankar'), ('Zaloha')
""")
mycursor.execute("""
    INSERT INTO hraci (jmeno, prijmeni, datum_narozeni, tym_id, pozice_id) VALUES
        ('Adam','Prvni','1996-06-02',2,2), ('Pavel','Druhy','2000-09-09',1,3),
        ('Kuba','Treti','1998-01-05',3,1), ('Josef','Ctvrty','1997-01-06',6,2),
        ('Jirka','Paty','1999-03-08',2,6)
""")
mydb.commit()

# TODO: FULL OUTER JOIN → hraci + pozice (přes UNION) →


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE hraci")
mycursor.execute("DROP TABLE pozice")
mycursor.execute("DROP TABLE tymy")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 5 — Kompletní úloha

Celý úkol je na vás od začátku do konce:

1. Připojte se k databázi.
2. Vytvořte 3 tabulky:
   - `ucitele` (`id`, `jmeno`, `prijmeni`, `predmet_id`)
   - `predmety` (`id`, `nazev`, `hodin_tydne`)
   - `ucebny` (`id`, `nazev`, `kapacita`, `predmet_id`)
3. Vložte:
   - **4 předměty** (např. Matematika, Fyzika, Chemie, Programování)
   - **5 učitelů** — jeden s `predmet_id`, který neexistuje
   - **4 učebny** — jednu s `predmet_id`, který neexistuje
4. Vypište:
   - **INNER JOIN** učitelé + předměty → pouze učitelé s platným předmětem
   - **LEFT JOIN** učitelé + předměty → všichni učitelé (i bez předmětu)
   - **JOIN tří tabulek** → učitelé + předměty + učebny (kde se předmět učí)
5. Smažte tabulky a odpojte se.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
--- INNER JOIN (učitelé + předměty) ---
...

--- LEFT JOIN (všichni učitelé + předměty) ---
...

--- JOIN tří tabulek (učitelé + předměty + učebny) ---
...
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# TODO: Celý úkol je na vás →


# Nezapomeňte na konci smazat tabulky a odpojit se!